# 00 — Generar archivos de mapeo estandarizados

Genera 4 archivos en `Insumos/Mapeo/` a partir de los grafos del RES
(`regional.graphml` / `nacional.graphml`), los sets y CSVs reales de otoole,
el archivo guía `Desarrollo Colombia regionalizado.xlsx` y el
`diccionario.xlsx` de correspondencias de FUELs:

1. **participaciones.xlsx** — formato ancho `Parameter | TECHNOLOGY | FUEL | Año | CA OR SO AN NE SE IN`,
   con plantilla de parámetros aditivos e importación de las participaciones existentes.
2. **mapeo_tech_fuel.xlsx** — existencia por región y renombres regionales.
3. **diccionario_fuel.xlsx** — correspondencia FUEL regional ↔ nacional enriquecida con los grafos.
4. **diccionario_tech.xlsx** — correspondencia de tecnologías con fuels IAR/OAR según los grafos.

Los grafos son la fuente primaria de topología (conexiones IAR/OAR); los CSVs de
otoole se usan como validación cruzada. Ejecutable de principio a fin con *Run All*.


## 1. Setup e imports

In [ ]:
# --- Setup: raíz del proyecto, imports y constantes ---
import re
import sys
import warnings
from collections import defaultdict
from pathlib import Path

import networkx as nx
import pandas as pd
import yaml

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)

RAIZ = Path.cwd().resolve()
if not (RAIZ / "config").exists():   # ejecutado desde notebooks/
    RAIZ = RAIZ.parent

# Prefijos canónicos (verificados contra CSV_Regional/FUEL.csv).
# Ojo: la región "Este" de los insumos corresponde a Sureste -> SE (no ES).
REGIONES = ["CA", "OR", "SO", "AN", "NE", "SE", "IN"]          # orden de columnas de salida
MAPEO_REGION = {
    "Antioquia": "AN", "Caribe": "CA", "Este": "SE", "Insular": "IN",
    "Nordeste": "NE", "Oriente": "OR", "Suroccidente": "SO",
}
RE_PREFIJO = re.compile(r"^(AN|CA|SE|IN|NE|OR|SO)_(.+)$")

DIR_MAPEO = RAIZ / "Insumos" / "Mapeo"
DIR_MAPEO.mkdir(parents=True, exist_ok=True)

RUTA_GRAPHML_REGIONAL = RAIZ / "regional.graphml"
RUTA_GRAPHML_NACIONAL = RAIZ / "nacional.graphml"
RUTA_GUIA        = RAIZ / "Insumos" / "Parametros_conexion" / "Desarrollo Colombia regionalizado.xlsx"
RUTA_DICCIONARIO = RAIZ / "Insumos" / "Parametros_conexion" / "diccionario.xlsx"

ALERTAS = defaultdict(list)   # {categoria: [mensajes]} -> se resumen en la sección 8


def split_prefijo(codigo):
    """'AN_MINAFR' -> ('AN', 'MINAFR'); sin prefijo -> (None, codigo)."""
    m = RE_PREFIJO.match(str(codigo))
    return (m.group(1), m.group(2)) if m else (None, str(codigo))


def unir(codigos):
    """Set de códigos -> 'A;B;C' ordenado."""
    return ";".join(sorted(codigos)) if codigos else ""


print(f"Raíz del proyecto : {RAIZ}")
print(f"Salidas           : {DIR_MAPEO}")


## 2a. Parsear los GraphML (fuente primaria de topología)

Nodos `type='technology'` → tecnologías; `type='fuel'` → fuels/commodities.
Aristas con `input_ratio` van fuel→tecnología (IAR); con `output_ratio`,
tecnología→fuel (OAR). Se ignoran las aristas `emission_ratio` y `Demand`
y los nodos `emission`/`demand` (el nodo `AnnualDemand` es un agregador).

Las aristas con ratio **0.0** se descartan: el nacional trae ~10.5k aristas
OAR de relleno con `output_ratio = 0` (p. ej. cada BACKSTOP conectado a todos
los fuels) que no son conexiones reales del RES.


In [2]:
def parsear_res(path_graphml, nombre):
    """Parsea un graphml del RES -> dict con sets de nodos y mapeos IAR/OAR."""
    G = nx.read_graphml(path_graphml)

    techs, fuels, sin_tipo = set(), set(), set()
    for n, d in G.nodes(data=True):
        t = d.get("type")
        if t == "technology":
            techs.add(n)
        elif t == "fuel":
            fuels.add(n)
        elif t in ("emission", "demand"):
            pass
        else:
            sin_tipo.add(n)

    iar = defaultdict(set)   # tech -> {fuels de entrada}
    oar = defaultdict(set)   # tech -> {fuels de salida}
    aristas_raras, n_cero = [], 0
    for u, v, d in G.edges(data=True):
        if "input_ratio" in d:            # fuel -> technology
            if d["input_ratio"] == 0:
                n_cero += 1
            elif v in techs:
                iar[v].add(u)
            else:
                aristas_raras.append((u, v, "input_ratio"))
        elif "output_ratio" in d:         # technology -> fuel
            if d["output_ratio"] == 0:
                n_cero += 1
            elif u in techs:
                oar[u].add(v)
            else:
                aristas_raras.append((u, v, "output_ratio"))
        # emission_ratio / Demand: fuera del alcance del mapeo

    n_iar = sum(len(s) for s in iar.values())
    n_oar = sum(len(s) for s in oar.values())
    print(f"[{nombre}] {len(techs)} nodos tecnología | {len(fuels)} nodos fuel | "
          f"{G.number_of_edges()} aristas ({n_iar} IAR, {n_oar} OAR, "
          f"{n_cero} con ratio 0 descartadas)")
    if sin_tipo:
        print(f"  ⚠ {len(sin_tipo)} nodos sin atributo 'type': {sorted(sin_tipo)[:10]}")
        ALERTAS[f"nodos_sin_type_{nombre}"] += sorted(sin_tipo)
    if aristas_raras:
        print(f"  ⚠ {len(aristas_raras)} aristas IAR/OAR cuyo extremo tecnología no es nodo technology")
        ALERTAS[f"aristas_raras_{nombre}"] += [f"{u}->{v} ({k})" for u, v, k in aristas_raras[:50]]
    return {"G": G, "techs": techs, "fuels": fuels, "sin_tipo": sin_tipo,
            "iar": dict(iar), "oar": dict(oar)}


grafo_reg = parsear_res(RUTA_GRAPHML_REGIONAL, "regional")

grafo_nac = None
if RUTA_GRAPHML_NACIONAL.exists():
    grafo_nac = parsear_res(RUTA_GRAPHML_NACIONAL, "nacional")
else:
    print(f"⚠ ADVERTENCIA: no existe {RUTA_GRAPHML_NACIONAL.name}; "
          "las conexiones nacionales se tomarán solo de CSV_Nacional/IAR+OAR.")
    ALERTAS["nacional_graphml"].append("nacional.graphml no encontrado: se usó CSV_Nacional como respaldo")


[regional] 2343 nodos tecnología | 541 nodos fuel | 6371 aristas (1964 IAR, 2361 OAR, 1 con ratio 0 descartadas)
  ⚠ 1 nodos sin atributo 'type': ['OIL002']


[nacional] 394 nodos tecnología | 113 nodos fuel | 16427 aristas (375 IAR, 399 OAR, 10478 con ratio 0 descartadas)


## 3. Cargar los demás insumos

Sets reales (`CSV_Regional`, `CSV_Nacional`), IAR/OAR de ambos modelos (validación
cruzada del grafo), `config_depurado.yaml` + `config/params_config.yaml` (parámetros
aditivos), `diccionario.xlsx`, el archivo guía y los diccionarios LEAP→OSeMOSYS de
los notebooks industriales.


In [ ]:
# --- Sets y CSVs reales ---
def cargar_set(path):
    return set(pd.read_csv(path)["VALUE"].astype(str))

techs_reg = cargar_set(RAIZ / "CSV_Regional" / "TECHNOLOGY.csv")
fuels_reg = cargar_set(RAIZ / "CSV_Regional" / "FUEL.csv")
techs_nac = cargar_set(RAIZ / "CSV_Nacional" / "TECHNOLOGY.csv")
fuels_nac = cargar_set(RAIZ / "CSV_Nacional" / "FUEL.csv")

def cargar_pares_csv(path):
    """IAR/OAR csv de otoole -> dict tech -> {fuels}. Igual que en el grafo, se
    descartan los pares cuyo ratio es 0 en todos los años (relleno sin conexión real)."""
    df = pd.read_csv(path, usecols=["TECHNOLOGY", "FUEL", "VALUE"])
    df = df[df["VALUE"] != 0][["TECHNOLOGY", "FUEL"]].drop_duplicates()
    pares = defaultdict(set)
    for t, f in df.itertuples(index=False):
        pares[str(t)].add(str(f))
    return dict(pares)

iar_csv_reg = cargar_pares_csv(RAIZ / "CSV_Regional" / "InputActivityRatio.csv")
oar_csv_reg = cargar_pares_csv(RAIZ / "CSV_Regional" / "OutputActivityRatio.csv")
iar_csv_nac = cargar_pares_csv(RAIZ / "CSV_Nacional" / "InputActivityRatio.csv")
oar_csv_nac = cargar_pares_csv(RAIZ / "CSV_Nacional" / "OutputActivityRatio.csv")

# Existencia por región y códigos base (sin prefijo) de los sets regionales reales
def existencia_por_region(codigos):
    """{'MINAFR': {'AN','CA',...}, ...} a partir de códigos con prefijo."""
    ex = defaultdict(set)
    sin_prefijo = set()
    for c in codigos:
        pref, base = split_prefijo(c)
        if pref:
            ex[base].add(pref)
        else:
            sin_prefijo.add(c)
    return dict(ex), sin_prefijo

exist_tech, techs_reg_sin_pref = existencia_por_region(techs_reg)
exist_fuel, fuels_reg_sin_pref = existencia_por_region(fuels_reg)
if techs_reg_sin_pref:
    ALERTAS["techs_regionales_sin_prefijo"] += sorted(techs_reg_sin_pref)
if fuels_reg_sin_pref:
    ALERTAS["fuels_regionales_sin_prefijo"] += sorted(fuels_reg_sin_pref)

# --- Parámetros aditivos: todos los 'param' de config_depurado menos los intensivos ---
cfg_otoole = yaml.safe_load(open(RAIZ / "config_depurado.yaml", encoding="utf-8"))
params_todos = [k for k, v in cfg_otoole.items() if isinstance(v, dict) and v.get("type") == "param"]
params_cfg = yaml.safe_load(open(RAIZ / "config" / "params_config.yaml", encoding="utf-8"))
intensivos = set(params_cfg.get("parametros_intensivos", []))
params_aditivos = sorted(p for p in params_todos if p not in intensivos)
print(f"{len(params_todos)} parámetros en config_depurado.yaml -> {len(params_aditivos)} aditivos")

# --- diccionario.xlsx (correspondencia de FUELs regional<->nacional) ---
df_dicc = pd.read_excel(RUTA_DICCIONARIO, sheet_name="Sheet")
print(f"diccionario.xlsx: {len(df_dicc)} filas")

# --- Archivo guía: escanear todas las hojas buscando códigos regionales <PREF>_<COD> ---
RE_CODIGO_REGIONAL = re.compile(r"^(AN|CA|SE|IN|NE|OR|SO)_[A-Z0-9][A-Za-z0-9_]*$")
codigos_guia = set()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    xl_guia = pd.ExcelFile(RUTA_GUIA)
    for hoja in xl_guia.sheet_names:
        df_h = xl_guia.parse(hoja, header=None, dtype=str)
        for col in df_h.columns:
            for val in df_h[col].dropna():
                val = str(val).strip()
                if RE_CODIGO_REGIONAL.match(val):
                    codigos_guia.add(val)
print(f"Archivo guía: {len(xl_guia.sheet_names)} hojas, {len(codigos_guia)} códigos regionales detectados")

# Nota: las participaciones industriales ya no requieren traductores LEAP -> OSeMOSYS:
# desde 2026-07 Participacion_Regional_Industrial.xlsx trae códigos OSeMOSYS directos
# (hojas Participacion_Fuel / Participacion_Technology, mismo formato que el residencial).

# --- Correspondencias FUEL nacional -> base regional fijadas a mano ---
# Mandan sobre lo que infiere `diccionario.xlsx`, cuyas filas de la familia ELC
# contradicen el uso real del modelo (decisión UPME, 2026-07-21). El criterio
# automático de la sección 5 llega al mismo resultado, así que quedan como
# anclaje explícito y documentado más que como excepción.
RENOMBRES_FUEL_MANUALES = {
    "ELC": "ELC001",
    "ELC002": "ELC003",
    "ELC003": "ELCEV001",
    "ELC004": "ELCEV002",
}
print(f"Correspondencias de FUEL fijadas a mano: {RENOMBRES_FUEL_MANUALES}")


## 4. Generar `participaciones.xlsx`

Formato ancho `Parameter | TECHNOLOGY | FUEL | Año | CA OR SO AN NE SE IN`.
`TECHNOLOGY`/`FUEL`/`Año` vacíos actúan como filtros opcionales (Año vacío =
aplica a todos los años). Se importan las participaciones existentes y se
agrega una fila-plantilla vacía por cada parámetro aditivo aún sin datos.


In [ ]:
COLS_PARTICIPACIONES = ["Parameter", "TECHNOLOGY", "FUEL", "Año"] + REGIONES


def a_formato_ancho(df, parametro, col_llave, llave_destino, traductor=None):
    """Hoja ancha por regiones-nombre (Fuel/Technology | Anio | Antioquia..Suroccidente)
    -> filas del formato participaciones. `traductor` mapea la llave a código OSeMOSYS
    (None = ya es código); llaves sin traducción se omiten con alerta."""
    filas, omitidas = [], set()
    cols_reg = [c for c in df.columns if c in MAPEO_REGION]
    for _, r in df.iterrows():
        llave = str(r[col_llave]).strip()
        if traductor is not None:
            if llave not in traductor:
                omitidas.add(llave)
                continue
            llave = traductor[llave]
        fila = {"Parameter": parametro, "TECHNOLOGY": "", "FUEL": "", "Año": r["Anio"]}
        fila[llave_destino] = llave
        for nombre in cols_reg:
            fila[MAPEO_REGION[nombre]] = pd.to_numeric(r[nombre], errors="coerce")
        filas.append(fila)
    if omitidas:
        ALERTAS[f"participaciones_sin_mapeo_{parametro}"] += sorted(omitidas)
        print(f"  ⚠ {parametro}: {len(omitidas)} llaves de origen sin mapeo a código OSeMOSYS (omitidas)")
    return pd.DataFrame(filas)


bloques = []

# 1) Industrial - hoja Participacion_Fuel (códigos OSeMOSYS: INDCLIM, INDDHT, ...)
#    -> AccumulatedAnnualDemand
df_ind_fuel = pd.read_excel(RAIZ / "Insumos" / "Participacion_Regional_Industrial.xlsx",
                            sheet_name="Participacion_Fuel")
bloques.append(a_formato_ancho(df_ind_fuel, "AccumulatedAnnualDemand", "Fuel", "FUEL"))

# 2) Industrial - hoja Participacion_Technology (códigos DEMIND*, eficiencias LOW/MID/HIG
#    agrupadas por familia) -> TotalTechnologyAnnualActivityLowerLimit
df_ind_tech = pd.read_excel(RAIZ / "Insumos" / "Participacion_Regional_Industrial.xlsx",
                            sheet_name="Participacion_Technology")
bloques.append(a_formato_ancho(df_ind_tech, "TotalTechnologyAnnualActivityLowerLimit",
                               "Technology", "TECHNOLOGY"))

# 3) Residencial: fuels (ya son códigos OSeMOSYS) y tecnologías (ídem)
df_res_fuel = pd.read_excel(RAIZ / "Insumos" / "Participacion_Fuel_RES.xlsx",
                            sheet_name="Participacion_Fuel")
bloques.append(a_formato_ancho(df_res_fuel, "AccumulatedAnnualDemand", "Fuel", "FUEL"))

df_res_tech = pd.read_excel(RAIZ / "Insumos" / "Participacion_Technology_RES.xlsx",
                            sheet_name="Participacion_Technology")
bloques.append(a_formato_ancho(df_res_tech, "TotalTechnologyAnnualActivityLowerLimit",
                               "Technology", "TECHNOLOGY"))

# 4) Participacion_Regional_Residencial.xlsx trae las mismas hojas Fuel/Technology;
#    se importan también y luego se eliminan duplicados exactos de llave.
df_rr = pd.ExcelFile(RAIZ / "Insumos" / "Participacion_Regional_Residencial.xlsx")
bloques.append(a_formato_ancho(df_rr.parse("Participacion_Fuel"),
                               "AccumulatedAnnualDemand", "Fuel", "FUEL"))
bloques.append(a_formato_ancho(df_rr.parse("Participacion_Technology"),
                               "TotalTechnologyAnnualActivityLowerLimit", "Technology", "TECHNOLOGY"))

df_part = pd.concat(bloques, ignore_index=True)
n_antes = len(df_part)
df_part = df_part.drop_duplicates(subset=["Parameter", "TECHNOLOGY", "FUEL", "Año"], keep="first")
print(f"Participaciones importadas: {len(df_part)} filas ({n_antes - len(df_part)} duplicados eliminados)")

# Validación: la suma por fila debe ser ~1 (se aceptan filas todo-cero: sin actividad)
suma = df_part[REGIONES].sum(axis=1)
malas = df_part[(suma - 1).abs().gt(1e-6) & suma.abs().gt(1e-6)]
if len(malas):
    print(f"⚠ {len(malas)} filas con suma de participaciones != 1 (y != 0)")
    ALERTAS["participaciones_suma_invalida"] += [
        f"{r.Parameter} | {r.TECHNOLOGY}{r.FUEL} | {r.Año}: suma={s:.6f}"
        for (_, r), s in zip(malas.iterrows(), df_part.loc[malas.index, REGIONES].sum(axis=1))
    ][:100]
else:
    print("Validación de sumas por fila: OK (≈1 o 0)")

# Plantilla vacía para los parámetros aditivos que aún no tienen participaciones
con_datos = set(df_part["Parameter"].unique())
plantilla = pd.DataFrame([{"Parameter": p, "TECHNOLOGY": "", "FUEL": "", "Año": ""}
                          for p in params_aditivos if p not in con_datos])
for c in REGIONES:
    plantilla[c] = pd.NA
df_part = pd.concat([plantilla, df_part], ignore_index=True)[COLS_PARTICIPACIONES]
df_part = df_part.sort_values(["Parameter", "FUEL", "TECHNOLOGY", "Año"], kind="stable",
                              na_position="last").reset_index(drop=True)

ruta_out = DIR_MAPEO / "participaciones.xlsx"
df_part.to_excel(ruta_out, index=False)
print(f"✔ {ruta_out.relative_to(RAIZ)}: {len(df_part)} filas "
      f"({len(plantilla)} filas-plantilla, {df_part['Parameter'].nunique()} parámetros)")
df_part.head(8)


## 5. Generar `mapeo_tech_fuel.xlsx`

Una fila por código nacional (tecnología o FUEL, sin prefijo) con la existencia
binaria por región según los sets reales `CSV_Regional`, corroborada con el
archivo guía y los nodos de `regional.graphml`. `TECHNOLOGY_REGIONAL` /
`FUEL_REGIONAL` solo se llenan cuando el nombre base cambia en el regional
(vacío = mismo nombre con prefijo estándar).

La correspondencia de FUELs **no** se puede leer directamente de
`diccionario.xlsx`: propone varias bases regionales para un mismo código nacional
(`ELC` → `ELC001` y `ELC003`; `OIL` → las tres `OIL001_*`), de modo que recorrerlo
sin más deja el resultado a merced del orden de las filas y permite que dos FUELs
nacionales caigan en el mismo código regional — al regionalizar ambos escribirían
`AN_ELC003` y se pisarían. Aquí se resuelve así:

1. `RENOMBRES_FUEL_MANUALES` (sección 3) manda sobre todo lo demás.
2. El resto se reparte por **evidencia**: solapamiento (Jaccard) de las tecnologías
   que consumen o producen cada fuel en los IAR/OAR reales, priorizando las bases
   que existen con prefijo de región — mandar un FUEL a una base solo-global
   (`URN`, `FOL`, `DSL001`…) equivaldría a dejarlo sin regionalizar.
3. Asignación greedy de mayor a menor, **1 a 1 por ambos lados**, con desempate
   alfabético para que sea reproducible. Sin evidencia de conexión no se inventa
   ningún renombramiento.

Un `assert` al final verifica que ningún par de FUELs nacionales comparta destino.


In [5]:
# --- Correspondencia FUEL nacional <-> base regional, 1 a 1 ---
# diccionario.xlsx propone varias bases regionales para un mismo FUEL nacional
# (ELC -> ELC001 y ELC003; OIL -> las tres OIL001_*), así que recorrerlo sin más
# deja el resultado a merced del orden de las filas y permite que dos FUELs
# nacionales aterricen en el mismo código regional (se pisarían al regionalizar).
# Se resuelve con evidencia: solapamiento de las tecnologías que consumen o
# producen cada fuel (Jaccard sobre IAR+OAR reales), asignación greedy de mayor
# a menor y unicidad por ambos lados.

def fuels_a_techs(*pares_por_tech):
    """{tech: {fuels}} -> {fuel base: {tech base}}, ambos sin prefijo."""
    out = defaultdict(set)
    for pares in pares_por_tech:
        for tech, fuels in pares.items():
            _, base_tech = split_prefijo(tech)
            for f in fuels:
                out[split_prefijo(f)[1]].add(base_tech)
    return out


techs_de_fuel_nac = fuels_a_techs(iar_csv_nac, oar_csv_nac)
techs_de_fuel_reg = fuels_a_techs(iar_csv_reg, oar_csv_reg)
bases_fuel_reg = set(exist_fuel) | fuels_reg_sin_pref


def solape_fuel(nac, reg):
    """Jaccard entre las tecnologías conectadas a un fuel nacional y a uno regional."""
    a, b = techs_de_fuel_nac.get(nac, set()), techs_de_fuel_reg.get(reg, set())
    return len(a & b) / len(a | b) if (a or b) else 0.0


# Candidatos por FUEL nacional: los que propone el diccionario, más la identidad
candidatos_fuel = defaultdict(set)
for _, r in df_dicc.dropna(subset=["fuel_regional", "fuel_nacional"]).iterrows():
    _, base_reg = split_prefijo(r["fuel_regional"])
    candidatos_fuel[str(r["fuel_nacional"]).strip()].add(base_reg)
for nac in fuels_nac:
    if nac in bases_fuel_reg:
        candidatos_fuel[nac].add(nac)

# Las decisiones manuales se fijan primero y sacan a ambos lados del reparto
mapa_fuel = {n: r for n, r in RENOMBRES_FUEL_MANUALES.items() if n in fuels_nac}
ocupadas = set(mapa_fuel.values())
for nac, reg in mapa_fuel.items():
    otros = candidatos_fuel.get(nac, set()) - {reg}
    if otros:
        ALERTAS["fuel_correspondencia_manual"].append(
            f"{nac} -> {reg} (manual; diccionario proponía {unir(otros)})")

# Reparto greedy. Ordena por (1) si la base existe con prefijo de región, (2) solape,
# (3) alfabético para ser determinista. El primer criterio importa porque el
# renombramiento se usa como f"{prefijo}_{base}": mandar un FUEL nacional a una base
# que solo existe global (URN, FOL, DSL001...) equivale a dejarlo sin regionalizar.
puntuadas = sorted(
    ((1 if exist_fuel.get(reg) else 0, solape_fuel(nac, reg), nac, reg)
     for nac, regs in candidatos_fuel.items() for reg in regs),
    key=lambda t: (-t[0], -t[1], t[2], t[3]))

for _, score, nac, reg in puntuadas:
    if nac in mapa_fuel or reg in ocupadas:
        if nac not in mapa_fuel and reg in ocupadas:
            ALERTAS["fuel_candidato_descartado"].append(
                f"{nac} -> {reg} (solape {score:.2f}): la base ya la tomó "
                f"'{[k for k, v in mapa_fuel.items() if v == reg][0]}'")
        continue
    if score == 0 and nac != reg:
        continue          # sin evidencia de conexión: no se inventa el renombramiento
    mapa_fuel[nac] = reg
    ocupadas.add(reg)
    if len(candidatos_fuel[nac]) > 1:
        detalle = ", ".join(f"{c}={solape_fuel(nac, c):.2f}"
                            for c in sorted(candidatos_fuel[nac]))
        ALERTAS["fuel_correspondencia_resuelta"].append(f"{nac} -> {reg} (solapes: {detalle})")

sin_correspondencia = sorted(fuels_nac - set(mapa_fuel))
for nac in sin_correspondencia:
    otros = candidatos_fuel.get(nac, set())
    ALERTAS["fuel_nacional_sin_base_regional"].append(
        f"{nac}" + (f" (candidatos sin evidencia de conexión: {unir(otros)})" if otros else ""))

# Correspondencias válidas pero que no producen códigos por región: el fuel existe
# global en el regional (commodity compartida), así que no se reparte.
no_regionalizables = sorted((n, r) for n, r in mapa_fuel.items() if not exist_fuel.get(r))
if no_regionalizables:
    ALERTAS["fuel_correspondencia_no_regionalizable"] += [
        f"{n} -> {r} (existe global, sin códigos por región)" for n, r in no_regionalizables]

# renombre_fuel se queda solo con los casos en que el nombre cambia
renombre_fuel = {n: r for n, r in mapa_fuel.items() if n != r}
renombre_fuel_inv = {r: n for n, r in mapa_fuel.items()}   # 1 a 1 por construcción
assert len(renombre_fuel_inv) == len(mapa_fuel), "la correspondencia de FUEL no es 1 a 1"
print(f"Correspondencias FUEL nacional -> base regional: {len(mapa_fuel)} "
      f"({len(renombre_fuel)} son renombramientos, "
      f"{len(ALERTAS.get('fuel_correspondencia_resuelta', []))} resueltas por evidencia, "
      f"{len(sin_correspondencia)} nacionales sin base regional)")
print("Renombramientos:", renombre_fuel)

def fila_existencia(bases_presentes):
    return {reg: (1 if reg in bases_presentes else 0) for reg in REGIONES}

filas = []
for tec in sorted(techs_nac):
    presentes = exist_tech.get(tec, set())
    filas.append({"TECHNOLOGY": tec, "FUEL": "", **fila_existencia(presentes),
                  "TECHNOLOGY_REGIONAL": "", "FUEL_REGIONAL": ""})
for fuel in sorted(fuels_nac):
    base_reg = renombre_fuel.get(fuel, fuel)
    presentes = exist_fuel.get(base_reg, set())
    filas.append({"TECHNOLOGY": "", "FUEL": fuel, **fila_existencia(presentes),
                  "TECHNOLOGY_REGIONAL": "",
                  "FUEL_REGIONAL": base_reg if base_reg != fuel else ""})

df_mapeo = pd.DataFrame(filas)[["TECHNOLOGY", "FUEL"] + REGIONES +
                               ["TECHNOLOGY_REGIONAL", "FUEL_REGIONAL"]]

# Complemento 1: nodos del grafo regional ausentes de los sets CSV -> alerta
grafo_no_csv_t = sorted(grafo_reg["techs"] - techs_reg)
grafo_no_csv_f = sorted(grafo_reg["fuels"] - fuels_reg)
if grafo_no_csv_t:
    print(f"⚠ {len(grafo_no_csv_t)} tecnologías en regional.graphml pero no en CSV_Regional/TECHNOLOGY.csv")
    ALERTAS["tech_grafo_no_en_csv"] += grafo_no_csv_t
if grafo_no_csv_f:
    print(f"⚠ {len(grafo_no_csv_f)} fuels en regional.graphml pero no en CSV_Regional/FUEL.csv")
    ALERTAS["fuel_grafo_no_en_csv"] += grafo_no_csv_f

# Complemento 2: códigos del archivo guía ausentes de los sets regionales -> alerta
guia_no_csv = sorted(codigos_guia - techs_reg - fuels_reg)
if guia_no_csv:
    print(f"⚠ {len(guia_no_csv)} códigos del archivo guía no existen en los sets regionales")
    ALERTAS["codigos_guia_no_en_csv"] += guia_no_csv

# Ningún par de FUELs nacionales puede aterrizar en el mismo código regional: si
# ocurriera, al regionalizar ambos escribirían p. ej. AN_ELC003 y se pisarían.
destinos = df_mapeo[df_mapeo["FUEL"] != ""].assign(
    destino=lambda d: d["FUEL_REGIONAL"].where(d["FUEL_REGIONAL"] != "", d["FUEL"]))
colisiones = destinos[destinos.duplicated("destino", keep=False)]
assert colisiones.empty, ("dos FUELs nacionales apuntan al mismo código regional:\n"
                          + colisiones[["FUEL", "FUEL_REGIONAL", "destino"]].to_string(index=False))
print("Sin colisiones: cada FUEL nacional apunta a un código regional distinto ✔")

ruta_out = DIR_MAPEO / "mapeo_tech_fuel.xlsx"
df_mapeo.to_excel(ruta_out, index=False)
print(f"✔ {ruta_out.relative_to(RAIZ)}: {len(df_mapeo)} filas "
      f"({len(techs_nac)} tecnologías + {len(fuels_nac)} fuels nacionales)")
df_mapeo.head(8)


Correspondencias FUEL nacional -> base regional: 98 (14 son renombramientos, 5 resueltas por evidencia, 15 nacionales sin base regional)
Renombramientos: {'ELC': 'ELC001', 'ELC002': 'ELC003', 'ELC003': 'ELCEV001', 'ELC004': 'ELCEV002', 'HDG': 'HDG001', 'COA': 'COA001', 'JET': 'JET002', 'BDL': 'BDL002', 'BET': 'BET002', 'NGS': 'NGS001', 'GSL': 'GSL003', 'DSL': 'DSL003', 'SAF': 'SAF001', 'LPG': 'LPG001'}
⚠ 1 fuels en regional.graphml pero no en CSV_Regional/FUEL.csv
⚠ 266 códigos del archivo guía no existen en los sets regionales
Sin colisiones: cada FUEL nacional apunta a un código regional distinto ✔


✔ Insumos\Mapeo\mapeo_tech_fuel.xlsx: 507 filas (394 tecnologías + 113 fuels nacionales)


,TECHNOLOGY,FUEL,CA,OR,SO,AN,NE,SE,IN,TECHNOLOGY_REGIONAL,FUEL_REGIONAL
0,BACKSTOP_1,,0,0,0,0,0,0,0,,
1,BACKSTOP_2,,0,0,0,0,0,0,0,,
2,BBGDST,,0,0,0,0,0,0,0,,
3,DEMAGFDSL,,1,1,1,1,1,1,1,,
4,DEMAGFELC,,1,1,1,1,1,1,1,,
5,DEMAGFGSL,,1,1,1,1,1,1,1,,
6,DEMAGFNGS,,1,1,1,1,1,0,0,,
7,DEMAGFWOO,,1,1,1,1,1,1,1,,


## 6. Generar `diccionario_fuel.xlsx`

**Una fila por FUEL base (sin prefijo)**: las columnas `CA..IN` indican en qué
regiones existe (`AN_AFR` en `CSV_Regional/FUEL.csv` ⇒ columna `AN = 1` en la
fila `AFR`), de modo que no se repite una fila por cada prefijo.

- Se parte de `CSV_Regional/FUEL.csv`; los códigos sin prefijo reconocido
  (p. ej. `URN`, `FOL`) quedan con base = el código completo y `CA..IN = 0`.
- `FUEL_NACIONAL` sale de `diccionario.xlsx` (se conserva aunque sea igual al
  regional, es decir, cuando no hubo renombramiento).
- **TECH_SHARED** — tecnologías conectadas a este FUEL en `regional.graphml`
  (unión de las regiones) cuyo nombre base también existe en el nacional.
- **TECH_NO_SHARED** — tecnologías conectadas al FUEL nacional (según
  `nacional.graphml`, o CSV_Nacional si no existe) sin correspondencia regional.
- Los FUELs nacionales sin ninguna base regional se incluyen igual, con las
  7 columnas en 0 y `OBSERVACION = "fuel sin prefijo regional"` (misma marca que
  reciben los códigos regionales sin prefijo).


In [6]:
# Conexiones fuel -> tecnologías (IAR + OAR) en cada modelo
def conexiones_por_fuel(iar, oar):
    con = defaultdict(set)
    for tech, fs in list(iar.items()) + list(oar.items()):
        for f in fs:
            con[f].add(tech)
    return dict(con)

techs_por_fuel_reg = conexiones_por_fuel(grafo_reg["iar"], grafo_reg["oar"])
if grafo_nac is not None:
    techs_por_fuel_nac = conexiones_por_fuel(grafo_nac["iar"], grafo_nac["oar"])
    fuente_nac = "nacional.graphml"
else:
    techs_por_fuel_nac = conexiones_por_fuel(iar_csv_nac, oar_csv_nac)
    fuente_nac = "CSV_Nacional (IAR+OAR)"
print(f"Conexiones nacionales tomadas de: {fuente_nac}")

bases_tech_reg = set(exist_tech)   # códigos base de tecnología presentes en el regional

OBS_SIN_PREFIJO = "fuel sin prefijo regional"

# --- 1-3) FUELs base a partir de CSV_Regional/FUEL.csv ---
# Un fuel con prefijo aporta su región a la fila de su base; uno sin prefijo
# reconocido queda como base propia sin ninguna región marcada.
nodos_reg_por_base = defaultdict(set)   # base -> {nombres reales en el modelo regional}
for f in fuels_reg:
    _, base = split_prefijo(f)
    nodos_reg_por_base[base].add(f)

# --- 5) base regional -> FUEL nacional ---
# La autoridad es la correspondencia 1 a 1 resuelta en la sección 5; así este
# archivo no puede contradecir a mapeo_tech_fuel.xlsx. Las bases que quedaron
# fuera del reparto no tienen equivalente nacional y se dejan en blanco.
base_a_nacional = dict(renombre_fuel_inv)

obs_por_base = defaultdict(list)
for _, r in df_dicc.dropna(subset=["fuel_regional"]).iterrows():
    _, base = split_prefijo(str(r["fuel_regional"]).strip())
    if not pd.isna(r.get("observacion")):
        obs = str(r["observacion"]).strip()
        if obs and obs not in obs_por_base[base]:
            obs_por_base[base].append(obs)


def construir_fila(base, nombres_regionales, f_nac, observaciones):
    """Una fila del diccionario de fuels a partir de la base sin prefijo."""
    regiones_presentes = {p for p in (split_prefijo(n)[0] for n in nombres_regionales) if p}

    # TECH_SHARED: unión de las tecnologías conectadas a cualquier variante regional
    shared = set()
    for nombre in nombres_regionales:
        for tech in techs_por_fuel_reg.get(nombre, set()):
            _, base_tech = split_prefijo(tech)
            if base_tech in techs_nac:
                shared.add(base_tech)
    # TECH_NO_SHARED: conectadas al fuel nacional y sin correspondencia regional
    no_shared = {t for t in techs_por_fuel_nac.get(f_nac, set())
                 if t not in bases_tech_reg} if f_nac else set()

    obs = list(observaciones)
    if not regiones_presentes:
        obs.insert(0, OBS_SIN_PREFIJO)
    return {"FUEL_REGIONAL": base, "FUEL_NACIONAL": f_nac,
            **fila_existencia(regiones_presentes),
            "TECH_SHARED": unir(shared), "TECH_NO_SHARED": unir(no_shared),
            "OBSERVACION": " | ".join(obs)}


# --- 4) una fila por base regional ---
filas, bases_huerfanas = [], []
for base in sorted(nodos_reg_por_base):
    # Sin fallback a la identidad: si la base no salió en el reparto 1 a 1 es que
    # no tiene equivalente nacional (su homónimo nacional ya está tomado por otra).
    f_nac = base_a_nacional.get(base, "")
    obs = list(obs_por_base.get(base, []))
    if not f_nac:
        bases_huerfanas.append(base)
        obs.append("SIN_EQUIVALENTE_NACIONAL")
    filas.append(construir_fila(base, nodos_reg_por_base[base], f_nac, obs))
if bases_huerfanas:
    print(f"⚠ {len(bases_huerfanas)} bases de FUEL regional sin equivalente nacional")
    ALERTAS["fuels_regionales_sin_equivalente"] += bases_huerfanas

# --- 6) FUELs nacionales que no corresponden a ninguna base regional ---
# (se excluyen los que ya son el nombre de una base, para no duplicar la fila)
nac_cubiertos = ({f["FUEL_NACIONAL"] for f in filas} |
                 {f["FUEL_REGIONAL"] for f in filas}) - {""}
fuels_nac_sin_corr = sorted(fuels_nac - nac_cubiertos)
for f_nac in fuels_nac_sin_corr:
    filas.append(construir_fila(f_nac, set(), f_nac, []))
if fuels_nac_sin_corr:
    print(f"⚠ {len(fuels_nac_sin_corr)} FUELs nacionales sin correspondencia regional "
          f"(incluidos con las 7 regiones en 0)")
    ALERTAS["fuels_nacionales_sin_correspondencia"] += fuels_nac_sin_corr

df_dicc_fuel = (pd.DataFrame(filas)[["FUEL_REGIONAL", "FUEL_NACIONAL"] + REGIONES +
                                    ["TECH_SHARED", "TECH_NO_SHARED", "OBSERVACION"]]
                .sort_values("FUEL_REGIONAL", kind="stable").reset_index(drop=True))

# Verificación del formato: una sola fila por base y sin prefijos en FUEL_REGIONAL
assert df_dicc_fuel["FUEL_REGIONAL"].is_unique, "hay bases de FUEL duplicadas"
con_prefijo = [f for f in df_dicc_fuel["FUEL_REGIONAL"] if RE_PREFIJO.match(f)]
assert not con_prefijo, f"FUEL_REGIONAL con prefijo: {con_prefijo[:5]}"

sin_region = int((df_dicc_fuel[REGIONES].sum(axis=1) == 0).sum())
ruta_out = DIR_MAPEO / "diccionario_fuel.xlsx"
df_dicc_fuel.to_excel(ruta_out, index=False)
print(f"✔ {ruta_out.relative_to(RAIZ)}: {len(df_dicc_fuel)} filas "
      f"(1 por FUEL base; {sin_region} sin prefijo regional)")
df_dicc_fuel.head(8)


Conexiones nacionales tomadas de: nacional.graphml


⚠ 16 bases de FUEL regional sin equivalente nacional
⚠ 15 FUELs nacionales sin correspondencia regional (incluidos con las 7 regiones en 0)
✔ Insumos\Mapeo\diccionario_fuel.xlsx: 129 filas (1 por FUEL base; 25 sin prefijo regional)


,FUEL_REGIONAL,FUEL_NACIONAL,CA,OR,SO,AN,NE,SE,IN,TECH_SHARED,TECH_NO_SHARED,OBSERVACION
0,AFR,AFR,1,1,1,1,1,1,0,MINAFR;PWRAFR,,
1,AGFELC,AGFELC,1,1,1,1,1,1,1,DEMAGFELC,,Firmas no idénticas
2,AGFHEA,AGFHEA,1,1,1,1,1,1,1,DEMAGFDSL;DEMAGFGSL;DEMAGFNGS;DEMAGFWOO,,Firmas no idénticas
3,BAG,BAG,0,1,1,0,0,0,0,DEMINDBAGBOI_HIG;DEMINDBAGBOI_LOW;DEMINDBAGBOI_MID;DEMINDBAGFURCCS;DEMINDBAG...,,Firmas no idénticas
4,BDL001,,0,0,0,0,0,0,0,UPSBDL,,fuel sin prefijo regional | Firmas no idénticas | SIN_EQUIVALENTE_NACIONAL
5,BDL002,BDL,1,1,1,1,1,0,0,UPSBDB,,Firmas no idénticas
6,BET001,,0,0,0,0,0,0,0,UPSBET,,fuel sin prefijo regional | Firmas no idénticas | SIN_EQUIVALENTE_NACIONAL
7,BET002,BET,1,1,1,1,1,0,0,UPSBGB,,Firmas no idénticas


## 7. Generar `diccionario_tech.xlsx`

Una fila por tecnología base regional (sin prefijo, excluyendo `TRN*` —
transmisión interregional nueva por diseño) más las nacionales sin equivalente.

Las columnas IAR/OAR **regionales** se construyen con los CSV como fuente primaria
de valores (filtrando nulos y ceros) y el grafo como validación estructural:

- fuels que traían prefijo reconocido (`AN_`, `CA_`, …) entran directo;
- fuels **sin** prefijo entran solo si el grafo respalda la arista — así se conservan
  los casos especiales legítimos (`URN`, `FOL`, `DSL001`…) y se descartan las
  conexiones espurias, que quedan registradas en la tabla de anomalías de la 7b;
- las aristas del grafo sin datos en el CSV se registran pero no se incluyen.

Las columnas **nacionales** salen solo de `CSV_Nacional/IAR+OAR` (el nacional no
tiene prefijos que filtrar); `nacional.graphml`, si existe, sirve únicamente para
detectar aristas sin datos.

`EQUIVALENCIA` compara ambos lados traduciendo los códigos regionales a
nomenclatura nacional (vía `diccionario.xlsx`), de modo que un renombramiento puro
(`SAF001` ↔ `SAF`) no cuente como diferencia; las columnas conservan los códigos
regionales reales.


In [7]:
# --- Lado regional: CSV como fuente primaria de valores, grafo como validación ---
def conexiones_csv_regional(path):
    """CSV IAR/OAR regional -> (prefijados, sin_prefijo), ambos dict tech_base -> {fuel_base}.
    Se descartan los valores nulos o 0 (relleno sin conexión real)."""
    df = pd.read_csv(path, usecols=["TECHNOLOGY", "FUEL", "VALUE"])
    df = df[df["VALUE"].notna() & (df["VALUE"] != 0)]
    prefijados, sin_prefijo = defaultdict(set), defaultdict(set)
    for t, f in df[["TECHNOLOGY", "FUEL"]].drop_duplicates().itertuples(index=False):
        _, base_tech = split_prefijo(t)
        pref_fuel, base_fuel = split_prefijo(f)
        destino = prefijados if pref_fuel else sin_prefijo
        destino[base_tech].add(base_fuel)
    return dict(prefijados), dict(sin_prefijo)


def agrupar_grafo_por_base(conexiones):
    """{tech con prefijo: {fuels con prefijo}} -> {tech base: {fuel base}}."""
    out = defaultdict(set)
    for tech, fuels in conexiones.items():
        _, base_tech = split_prefijo(tech)
        out[base_tech] |= {split_prefijo(f)[1] for f in fuels}
    return dict(out)


iar_pre_reg, iar_sin_reg = conexiones_csv_regional(RAIZ / "CSV_Regional" / "InputActivityRatio.csv")
oar_pre_reg, oar_sin_reg = conexiones_csv_regional(RAIZ / "CSV_Regional" / "OutputActivityRatio.csv")
iar_grafo_reg = agrupar_grafo_por_base(grafo_reg["iar"])
oar_grafo_reg = agrupar_grafo_por_base(grafo_reg["oar"])

anomalias = []   # TECHNOLOGY | FUEL | IAR_o_OAR | FUENTE | MOTIVO


def resolver_regional(base, prefijados, sin_prefijo, grafo, etiqueta):
    """Fuels regionales confirmados para una tecnología base.

    Los que traían prefijo reconocido entran directo; los que no, solo si el grafo
    respalda la arista (casos especiales legítimos como URN o FOL). Lo demás se
    registra como anomalía y NO entra en la columna."""
    del_csv = set(prefijados.get(base, set()))
    del_grafo = grafo.get(base, set())
    confirmados = set(del_csv)

    for fuel in sin_prefijo.get(base, set()):
        if fuel in del_grafo:
            confirmados.add(fuel)          # caso especial legítimo sin prefijo
        else:
            anomalias.append({"TECHNOLOGY": base, "FUEL": fuel, "IAR_o_OAR": etiqueta,
                              "FUENTE": "solo_en_csv",
                              "MOTIVO": "sin prefijo y sin arista en grafo"})
    for fuel in del_grafo - del_csv - sin_prefijo.get(base, set()):
        anomalias.append({"TECHNOLOGY": base, "FUEL": fuel, "IAR_o_OAR": etiqueta,
                          "FUENTE": "solo_en_grafo",
                          "MOTIVO": "arista en grafo sin datos en CSV"})
    return confirmados


# --- Lado nacional: solo CSV (no hay prefijos que filtrar) ---
iar_nac, oar_nac = iar_csv_nac, oar_csv_nac
NOTA_SIN_GRAFO_NAC = "nacional.graphml ausente: conexiones nacionales sin validación estructural"
if grafo_nac is not None:
    for etiqueta, grafo_n, csv_n in [("IAR", grafo_nac["iar"], iar_nac),
                                     ("OAR", grafo_nac["oar"], oar_nac)]:
        for tech, fuels in grafo_n.items():
            for fuel in fuels - csv_n.get(tech, set()):
                anomalias.append({"TECHNOLOGY": tech, "FUEL": fuel, "IAR_o_OAR": etiqueta,
                                  "FUENTE": "solo_en_grafo",
                                  "MOTIVO": "arista en grafo nacional sin datos en CSV"})
else:
    print(f"⚠ {NOTA_SIN_GRAFO_NAC}")


def a_nomenclatura_nacional(fuels_base):
    """Aplica los renombramientos del diccionario (ELC003 -> ELC) SOLO para comparar
    ambos modelos; las columnas conservan los códigos regionales reales."""
    return {renombre_fuel_inv.get(f, f) for f in fuels_base}


filas = []
bases_regional = sorted(b for b in exist_tech if not b.startswith("TRN"))
n_trn = sum(1 for b in exist_tech if b.startswith("TRN"))
print(f"{len(bases_regional)} tecnologías base regionales ({n_trn} TRN* excluidas)")

for base in bases_regional:
    en_nacional = base in techs_nac
    iar_r = resolver_regional(base, iar_pre_reg, iar_sin_reg, iar_grafo_reg, "IAR")
    oar_r = resolver_regional(base, oar_pre_reg, oar_sin_reg, oar_grafo_reg, "OAR")
    iar_n = set(iar_nac.get(base, set())) if en_nacional else set()
    oar_n = set(oar_nac.get(base, set())) if en_nacional else set()
    if not en_nacional:
        equiv = "SIN_CORRESPONDENCIA"
        obs = "Tecnología regional sin equivalente nacional"
    else:
        # La comparación traduce los códigos regionales a nomenclatura nacional para
        # que un simple renombramiento (SAF001 -> SAF) no cuente como diferencia.
        iar_rc, oar_rc = a_nomenclatura_nacional(iar_r), a_nomenclatura_nacional(oar_r)
        equiv = bool(iar_rc == iar_n and oar_rc == oar_n)
        obs = ""
        if equiv is False:
            difs = []
            if iar_rc != iar_n:
                difs.append(f"IAR solo regional: {unir(iar_rc - iar_n) or '-'} | "
                            f"solo nacional: {unir(iar_n - iar_rc) or '-'}")
            if oar_rc != oar_n:
                difs.append(f"OAR solo regional: {unir(oar_rc - oar_n) or '-'} | "
                            f"solo nacional: {unir(oar_n - oar_rc) or '-'}")
            obs = " ; ".join(difs)
        if grafo_nac is None:
            obs = f"{obs} ; {NOTA_SIN_GRAFO_NAC}" if obs else NOTA_SIN_GRAFO_NAC
    filas.append({"TECHNOLOGY_REGIONAL": base,
                  "TECHNOLOGY_NACIONAL": base if en_nacional else "",
                  **fila_existencia(exist_tech.get(base, set())),
                  "IAR_FUEL_REGIONAL": unir(iar_r), "IAR_FUEL_NACIONAL": unir(iar_n),
                  "OAR_FUEL_REGIONAL": unir(oar_r), "OAR_FUEL_NACIONAL": unir(oar_n),
                  "EQUIVALENCIA": equiv, "OBSERVACION": obs})

# Tecnologías nacionales que no existen en ninguna región
techs_nac_sin_corr = sorted(techs_nac - set(bases_regional))
for base in techs_nac_sin_corr:
    filas.append({"TECHNOLOGY_REGIONAL": "", "TECHNOLOGY_NACIONAL": base,
                  **{reg: 0 for reg in REGIONES},
                  "IAR_FUEL_REGIONAL": "", "IAR_FUEL_NACIONAL": unir(iar_nac.get(base, set())),
                  "OAR_FUEL_REGIONAL": "", "OAR_FUEL_NACIONAL": unir(oar_nac.get(base, set())),
                  "EQUIVALENCIA": "SIN_CORRESPONDENCIA",
                  "OBSERVACION": "Tecnología nacional sin equivalente regional"})
if techs_nac_sin_corr:
    print(f"⚠ {len(techs_nac_sin_corr)} tecnologías nacionales sin correspondencia regional")
    ALERTAS["techs_nacionales_sin_correspondencia"] += techs_nac_sin_corr

df_dicc_tech = pd.DataFrame(filas)[
    ["TECHNOLOGY_REGIONAL", "TECHNOLOGY_NACIONAL"] + REGIONES +
    ["IAR_FUEL_REGIONAL", "IAR_FUEL_NACIONAL", "OAR_FUEL_REGIONAL", "OAR_FUEL_NACIONAL",
     "EQUIVALENCIA", "OBSERVACION"]]

resumen_eq = df_dicc_tech["EQUIVALENCIA"].astype(str).value_counts()
print("EQUIVALENCIA:", dict(resumen_eq))

ruta_out = DIR_MAPEO / "diccionario_tech.xlsx"
df_dicc_tech.to_excel(ruta_out, index=False)
print(f"✔ {ruta_out.relative_to(RAIZ)}: {len(df_dicc_tech)} filas")
df_dicc_tech.head(8)


481 tecnologías base regionales (0 TRN* excluidas)
⚠ 28 tecnologías nacionales sin correspondencia regional
EQUIVALENCIA: {'True': np.int64(302), 'SIN_CORRESPONDENCIA': np.int64(143), 'False': np.int64(64)}


✔ Insumos\Mapeo\diccionario_tech.xlsx: 509 filas


,TECHNOLOGY_REGIONAL,TECHNOLOGY_NACIONAL,CA,OR,SO,AN,NE,SE,IN,IAR_FUEL_REGIONAL,IAR_FUEL_NACIONAL,OAR_FUEL_REGIONAL,OAR_FUEL_NACIONAL,EQUIVALENCIA,OBSERVACION
0,BACKSTOP_AGFELC,,1,1,1,1,1,1,1,,,AGFELC,,SIN_CORRESPONDENCIA,Tecnología regional sin equivalente nacional
1,BACKSTOP_AGFHEA,,1,1,1,1,1,1,1,,,AGFHEA,,SIN_CORRESPONDENCIA,Tecnología regional sin equivalente nacional
2,BACKSTOP_INDCLIM,,1,1,1,1,1,0,0,,,INDCLIM,,SIN_CORRESPONDENCIA,Tecnología regional sin equivalente nacional
3,BACKSTOP_INDDHT,,1,1,1,1,1,0,0,,,INDDHT,,SIN_CORRESPONDENCIA,Tecnología regional sin equivalente nacional
4,BACKSTOP_INDIHT,,1,1,1,1,1,0,0,,,INDIHT,,SIN_CORRESPONDENCIA,Tecnología regional sin equivalente nacional
5,BACKSTOP_INDILU,,1,1,1,1,1,0,0,,,INDILU,,SIN_CORRESPONDENCIA,Tecnología regional sin equivalente nacional
6,BACKSTOP_INDMPW,,1,1,1,1,1,0,0,,,INDMPW,,SIN_CORRESPONDENCIA,Tecnología regional sin equivalente nacional
7,BACKSTOP_INDOTH_COA,,1,1,1,1,1,0,0,,,INDOTH_COA,,SIN_CORRESPONDENCIA,Tecnología regional sin equivalente nacional


### 7b. Validación de las conexiones IAR/OAR

Anomalías detectadas al cruzar CSV y grafo, tecnologías que se quedaron sin
conexiones regionales pese a tener datos en el CSV, y magnitud de la
divergencia regional vs nacional.

> Esta tabla trabaja **por código base**, mientras que el cruce de la sección 8
> compara los pares con prefijo. Una arista presente en una sola región puede
> aparecer allí como `solo grafo` y no ser anomalía aquí, porque otra región sí
> aporta esa conexión en el CSV para la misma base.


In [8]:
df_anomalias = pd.DataFrame(anomalias, columns=["TECHNOLOGY", "FUEL", "IAR_o_OAR",
                                                "FUENTE", "MOTIVO"])
if len(df_anomalias):
    df_anomalias = df_anomalias.drop_duplicates().sort_values(
        ["FUENTE", "IAR_o_OAR", "TECHNOLOGY", "FUEL"]).reset_index(drop=True)
    ALERTAS["conexiones_anomalas"] += [
        f"{r.TECHNOLOGY} | {r.FUEL} | {r.IAR_o_OAR} | {r.FUENTE} | {r.MOTIVO}"
        for r in df_anomalias.itertuples(index=False)]

print(f"ANOMALÍAS DE CONEXIÓN: {len(df_anomalias)}")
if len(df_anomalias):
    print(df_anomalias.groupby(["FUENTE", "IAR_o_OAR"]).size().to_string())
    print()
    print(df_anomalias.to_string(index=False))
else:
    print("Ninguna: CSV y grafo coinciden en todas las conexiones. ✔")

# Tecnologías con datos en el CSV cuya columna regional quedó vacía
print()
for etiqueta, col, pre, sin_p in [("IAR", "IAR_FUEL_REGIONAL", iar_pre_reg, iar_sin_reg),
                                  ("OAR", "OAR_FUEL_REGIONAL", oar_pre_reg, oar_sin_reg)]:
    con_datos_csv = {b for b in set(pre) | set(sin_p) if (pre.get(b) or sin_p.get(b))}
    vacias = sorted(con_datos_csv & set(
        df_dicc_tech.loc[df_dicc_tech[col] == "", "TECHNOLOGY_REGIONAL"]))
    print(f"{etiqueta}: {len(vacias)} tecnologías con datos en CSV pero {col} vacío"
          + (f" -> {vacias[:15]}" if vacias else ""))
    if vacias:
        ALERTAS[f"{col}_vacio_con_datos_csv"] += vacias

# Divergencia regional vs nacional (solo donde hay correspondencia)
print()
comparables = df_dicc_tech[df_dicc_tech["EQUIVALENCIA"] != "SIN_CORRESPONDENCIA"]
for etiqueta in ["IAR", "OAR"]:
    distintas = (comparables[f"{etiqueta}_FUEL_REGIONAL"]
                 != comparables[f"{etiqueta}_FUEL_NACIONAL"]).sum()
    print(f"{etiqueta}_FUEL_REGIONAL != {etiqueta}_FUEL_NACIONAL: "
          f"{distintas} de {len(comparables)} tecnologías comparables "
          "(incluye renombramientos como ELC003/ELC002)")
df_anomalias.head(20)


ANOMALÍAS DE CONEXIÓN: 0
Ninguna: CSV y grafo coinciden en todas las conexiones. ✔

IAR: 0 tecnologías con datos en CSV pero IAR_FUEL_REGIONAL vacío
OAR: 0 tecnologías con datos en CSV pero OAR_FUEL_REGIONAL vacío

IAR_FUEL_REGIONAL != IAR_FUEL_NACIONAL: 223 de 366 tecnologías comparables (incluye renombramientos como ELC003/ELC002)
OAR_FUEL_REGIONAL != OAR_FUEL_NACIONAL: 45 de 366 tecnologías comparables (incluye renombramientos como ELC003/ELC002)


,TECHNOLOGY,FUEL,IAR_o_OAR,FUENTE,MOTIVO


## 8. Resumen final y alertas

Validación cruzada grafo ↔ CSV (aristas del grafo sin fila IAR/OAR y viceversa)
y consolidado de todas las alertas acumuladas durante la generación.


In [9]:
# --- Validación cruzada: pares (tecnología, fuel) del grafo vs CSVs de otoole ---
def pares(dic):
    return {(t, f) for t, fs in dic.items() for f in fs}

def validar_cruzado(nombre, grafo_iar, grafo_oar, csv_iar, csv_oar):
    for etiqueta, g, c in [("IAR", grafo_iar, csv_iar), ("OAR", grafo_oar, csv_oar)]:
        pg, pc = pares(g), pares(c)
        solo_grafo, solo_csv = sorted(pg - pc), sorted(pc - pg)
        print(f"[{nombre} {etiqueta}] grafo={len(pg)} pares | csv={len(pc)} | "
              f"solo grafo={len(solo_grafo)} | solo csv={len(solo_csv)}")
        if solo_grafo:
            ALERTAS[f"{nombre}_{etiqueta}_solo_en_grafo"] += [f"{t} <- {f}" for t, f in solo_grafo[:100]]
        if solo_csv:
            ALERTAS[f"{nombre}_{etiqueta}_solo_en_csv"] += [f"{t} <- {f}" for t, f in solo_csv[:100]]

validar_cruzado("regional", grafo_reg["iar"], grafo_reg["oar"], iar_csv_reg, oar_csv_reg)
if grafo_nac is not None:
    validar_cruzado("nacional", grafo_nac["iar"], grafo_nac["oar"], iar_csv_nac, oar_csv_nac)

# --- Consolidado de alertas ---
print()
print("=" * 70)
print("ARCHIVOS GENERADOS EN Insumos/Mapeo/:")
for f in ["participaciones.xlsx", "mapeo_tech_fuel.xlsx",
          "diccionario_fuel.xlsx", "diccionario_tech.xlsx"]:
    p = DIR_MAPEO / f
    print(f"  ✔ {f} ({p.stat().st_size / 1024:.0f} KB)")

print()
if not ALERTAS:
    print("Sin alertas. ✔")
else:
    print(f"ALERTAS ({sum(len(v) for v in ALERTAS.values())} en {len(ALERTAS)} categorías):")
    df_alertas = pd.DataFrame(
        [{"Categoria": cat, "N": len(msgs), "Ejemplos": "; ".join(map(str, msgs[:5]))}
         for cat, msgs in sorted(ALERTAS.items())])
    with pd.option_context("display.max_colwidth", 120):
        print(df_alertas.to_string(index=False))
    # Detalle completo a Excel para revisión
    ruta_alertas = DIR_MAPEO / "alertas_generacion.xlsx"
    pd.DataFrame([{"Categoria": cat, "Detalle": m} for cat, msgs in sorted(ALERTAS.items())
                  for m in msgs]).to_excel(ruta_alertas, index=False)
    print(f"Detalle completo: {ruta_alertas.relative_to(RAIZ)}")


[regional IAR] grafo=1964 pares | csv=1957 | solo grafo=8 | solo csv=1
[regional OAR] grafo=2361 pares | csv=2361 | solo grafo=0 | solo csv=0
[nacional IAR] grafo=375 pares | csv=375 | solo grafo=0 | solo csv=0
[nacional OAR] grafo=399 pares | csv=399 | solo grafo=0 | solo csv=0

ARCHIVOS GENERADOS EN Insumos/Mapeo/:
  ✔ participaciones.xlsx (508 KB)
  ✔ mapeo_tech_fuel.xlsx (22 KB)
  ✔ diccionario_fuel.xlsx (13 KB)
  ✔ diccionario_tech.xlsx (32 KB)

ALERTAS (550 en 17 categorías):
                                                        Categoria   N                                                                                                                                                                                                                                                                                                             Ejemplos
                                           codigos_guia_no_en_csv 266                                                                 

Detalle completo: Insumos\Mapeo\alertas_generacion.xlsx
